# NovaMart Retail: Round 3 Recommendation Engine, Action Cards & What-If
**StockSense Decision-Support System** — IntelliData 2026 Data Science Hackathon

This notebook implements Phase 5 (Round 3):
- Automated 7-Day Demand Forecast & Dynamic Safety Stock Calculation
- Priority Manager Action Queue (Sorted by Risk Tier & Revenue at Risk)
- Store Manager Recommendation Cards with Plain-English "Why?" Driver Explanations
- Interactive Scenario Simulator (What-If Analysis for Markdowns, Delays & Festival Spikes)


In [1]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.recommend import score_inventory_recommendations, get_recommendation_card, what_if_scenario

scored_df, manager_table = score_inventory_recommendations()
print(f"Generated recommendations for {len(manager_table):,} test-period observations.")

## 1. Top Priority Manager Action Queue
Filtered to High-Risk replenishment orders sorted by Revenue at Risk:


In [2]:
high_risk = manager_table[manager_table['risk_tier'] == 'High'].head(15)
display(high_risk[['store_id', 'store_name', 'product_name', 'current_stock', 'forecast_demand_7d', 'stockout_prob', 'risk_tier', 'recommended_order', 'revenue_at_risk']])

## 2. Interactive Store Manager Recommendation Card
Renders complete diagnostic and actionable directive for Store S01 x Product P101:


In [3]:
card = get_recommendation_card('S01', 'P101', scored_df=scored_df)
print(card['formatted_text'])

## 3. What-If Scenario Simulator
Simulates the commercial and operational impact of an unannounced supplier delay (+2 lead days) and promotional discount (+10% markdown):


In [4]:
sim = what_if_scenario('S01', 'P101', as_of_date=card['as_of_date'], discount_delta=10.0, extra_lead_days=2, festival_uplift_pct=15.0)
display(pd.DataFrame([
    {'Metric': '7-Day Demand Forecast', 'Baseline': sim['baseline']['forecast_7d'], 'Simulated': sim['simulated']['forecast_7d'], 'Net Shift': sim['net_impact']['demand_change']},
    {'Metric': 'Stock-out Probability', 'Baseline': f"{sim['baseline']['stockout_prob']*100:.1f}%", 'Simulated': f"{sim['simulated']['stockout_prob']*100:.1f}%", 'Net Shift': f"{sim['net_impact']['prob_change']*100:+.1f} pts"},
    {'Metric': 'Assigned Risk Tier', 'Baseline': sim['baseline']['risk_tier'], 'Simulated': sim['simulated']['risk_tier'], 'Net Shift': 'Shift' if sim['baseline']['risk_tier'] != sim['simulated']['risk_tier'] else 'Unchanged'},
    {'Metric': 'Recommended Reorder (Units)', 'Baseline': sim['baseline']['reorder_qty'], 'Simulated': sim['simulated']['reorder_qty'], 'Net Shift': f"{sim['net_impact']['reorder_qty_change']:+d} units"}
]))